# ML4SCI DeepLense: Common Test I - Multi-Class Classification

**Author:** Muntasir Mamun  
**Organization:** Machine Learning for Science (ML4SCI)  
**Project:** DeepLense (Gravitational Lensing Classification)  

**Task:** Build a deep learning model to classify strong gravitational lensing images into three classes:
- **No Substructure** (`no`)
- **Subhalo Substructure** (`sub`)
- **Vortex Substructure** (`vortex`)

**Evaluation Metrics:** Multi-Class ROC Curves and Area Under the Curve (AUC) on a 90:10 stratified split.


In [ ]:
# Step 1: Install & Import Dependencies
!pip install -q gdown scikit-learn matplotlib seaborn torch torchvision

import os
import glob
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix
from sklearn.preprocessing import label_binarize

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using acceleration device: {device}')


In [ ]:
# Step 2: Download & Extract Dataset from Official Google Drive
import gdown

file_id = '1ZEyNMEO43u3qhJAwJeBZxFBEYc_pVYZQ'
url = f'https://drive.google.com/uc?id={file_id}'
output_zip = 'dataset.zip'
data_root = './deeplense_data'

if not os.path.exists(output_zip) or os.path.getsize(output_zip) == 0:
    print('Downloading official DeepLense Common Test dataset...')
    gdown.download(url, output_zip, quiet=False)

if not os.path.exists(data_root) or len(os.listdir(data_root)) == 0:
    print('Extracting dataset...')
    !unzip -q {output_zip} -d {data_root}
    print('Extraction complete!')


In [ ]:
# Step 3: Robust Recursive Dataset Indexing & 90:10 Stratified Split
# Recursively find all .npy files, ignoring __MACOSX and hidden files
raw_files = glob.glob(os.path.join(data_root, '**', '*.npy'), recursive=True)
all_files = [f for f in raw_files if '__MACOSX' not in f and not os.path.basename(f).startswith('.')]

# Extract unique class names from parent directory of each file
classes = sorted(list(set(os.path.basename(os.path.dirname(f)) for f in all_files)))
print(f'Detected classes ({len(classes)}): {classes}')

all_filepaths = []
all_labels = []

for f in all_files:
    cls_name = os.path.basename(os.path.dirname(f))
    all_filepaths.append(f)
    all_labels.append(classes.index(cls_name))

for idx, cls in enumerate(classes):
    count = sum(1 for l in all_labels if l == idx)
    print(f'Class {cls}: {count} samples')

print(f'Total valid samples: {len(all_filepaths)}')

# Official 90:10 Stratified Split
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_filepaths, all_labels, test_size=0.10, random_state=42, stratify=all_labels
)

print(f'Train split: {len(train_paths)} (90%) | Validation split: {len(val_paths)} (10%)')


In [ ]:
# Step 4: Custom PyTorch Dataset with Rotational & Reflective Invariance
class GravitationalLensDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        path = self.filepaths[idx]
        label = self.labels[idx]

        arr = np.load(path)
        arr = np.squeeze(arr)
        # Min-Max Normalization to [0, 1]
        if arr.max() > arr.min():
            arr = (arr - arr.min()) / (arr.max() - arr.min())
        arr = (arr * 255).astype(np.uint8)
        if arr.ndim == 2:
            img = Image.fromarray(arr).convert('RGB')
        else:
            img = Image.fromarray(arr)

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.long)

# Gravitational lenses have rotational symmetry
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=90),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = GravitationalLensDataset(train_paths, train_labels, transform=train_transform)
val_dataset = GravitationalLensDataset(val_paths, val_labels, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
# Step 5: Pretrained ResNet-50 Model & Optimization
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Fine-tuning classification head
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(in_features, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(inplace=True),
    nn.Dropout(0.3),
    nn.Linear(256, len(classes))
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)


In [ ]:
# Step 6: Training & Validation Loop
epochs = 10
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outs = model(imgs)
        loss = criterion(outs, lbls)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outs, 1)
        correct += (preds == lbls).sum().item()
        total += lbls.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            outs = model(imgs)
            loss = criterion(outs, lbls)
            val_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outs, 1)
            val_correct += (preds == lbls).sum().item()
            val_total += lbls.size(0)

    scheduler.step()
    val_loss = val_loss / val_total
    val_acc = val_correct / val_total

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f'Epoch [{epoch+1:02d}/{epochs:02d}] Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'deeplense_resnet50_best.pth')

print(f'
Training complete! Best Val Loss: {best_val_loss:.4f}')


In [ ]:
# Step 7: Plot Learning Curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss', marker='o')
plt.plot(history['val_loss'], label='Val Loss', marker='s')
plt.title('Cross-Entropy Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Acc', marker='o')
plt.plot(history['val_acc'], label='Val Acc', marker='s')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('learning_curves.png', dpi=300)
plt.show()


In [ ]:
# Step 8: Multi-Class ROC Curves & AUC Score
model.load_state_dict(torch.load('deeplense_resnet50_best.pth'))
model.eval()

y_preds, y_trues = [], []

with torch.no_grad():
    for imgs, lbls in val_loader:
        imgs = imgs.to(device)
        outs = model(imgs)
        probs = torch.softmax(outs, dim=1).cpu().numpy()
        y_preds.append(probs)
        y_trues.append(lbls.numpy())

y_score = np.vstack(y_preds)
y_true = np.concatenate(y_trues)
y_true_bin = label_binarize(y_true, classes=range(len(classes)))

fpr = dict()
tpr = dict()
roc_auc = dict()

plt.figure(figsize=(8, 6))
colors = ['#1f77b4', '#d62728', '#2ca02c']

for i in range(len(classes)):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    plt.plot(fpr[i], tpr[i], color=colors[i], lw=2.5,
             label=f'ROC {classes[i]} (AUC = {roc_auc[i]:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Chance (AUC = 0.50)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('DeepLense Common Test I: Multi-Class ROC Curves', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.savefig('roc_curves.png', dpi=300)
plt.show()

macro_auc = np.mean(list(roc_auc.values()))
print('
==========================================')
print(f'Final Macro-Average ROC-AUC Score: {macro_auc:.4f}')
print('==========================================')
for i in range(len(classes)):
    print(f'  - Class {classes[i]}: AUC = {roc_auc[i]:.4f}')

with open('results_summary.txt', 'w') as f:
    f.write(f'Macro-Average ROC-AUC: {macro_auc:.4f}
')
    for i in range(len(classes)):
        f.write(f'Class {classes[i]} AUC: {roc_auc[i]:.4f}
')


In [ ]:
# Step 9: Confusion Matrix & Classification Report
preds_discrete = np.argmax(y_score, axis=1)
cm = confusion_matrix(y_true, preds_discrete)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

report = classification_report(y_true, preds_discrete, target_names=classes)
print('
Classification Report:')
print(report)

with open('results_summary.txt', 'a') as f:
    f.write('
Classification Report:
' + report + '
')
